#### The CelesTrack Dataset is very clean already, but mistaked can be happened by the dataset operators
Such as : 
- Missing values
- Duplication in the satellites
- value inconsistencies, etc

## Performing preprocssing on the satellite data
- Handle missing values
- Remove duplicates
- Structure data


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [4]:
df = pd.read_csv('../data/01_raw/gp.csv')

In [5]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-10-05T10:37:28.191360,13.762289,0.002461,90.2163,65.8752,347.2003,74.9038,0,U,900,999,3638,0.001091,1.074000e-05,0.0
1,CALSPHERE 2,1964-063E,2025-10-05T11:04:00.444864,13.528741,0.001678,90.2292,69.7816,244.2673,178.7129,0,U,902,999,82182,0.000110,8.200000e-07,0.0
2,LCS 1,1965-034C,2025-10-05T12:08:01.406976,9.893092,0.001358,32.1425,61.7207,300.3713,59.5442,0,U,1361,999,18466,-0.001468,-3.000000e-08,0.0
3,TEMPSAT 1,1965-065E,2025-10-05T06:44:31.462944,13.335738,0.006776,89.9871,212.7305,243.9611,271.6250,0,U,1512,999,92607,0.000109,6.300000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-10-05T13:13:59.657952,13.362208,0.007127,89.9109,124.8412,101.8063,291.1203,0,U,1520,999,92870,0.000297,1.650000e-06,0.0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12796 entries, 0 to 12795
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   OBJECT_NAME          12796 non-null  object 
 1   OBJECT_ID            12796 non-null  object 
 2   EPOCH                12796 non-null  object 
 3   MEAN_MOTION          12796 non-null  float64
 4   ECCENTRICITY         12796 non-null  float64
 5   INCLINATION          12796 non-null  float64
 6   RA_OF_ASC_NODE       12796 non-null  float64
 7   ARG_OF_PERICENTER    12796 non-null  float64
 8   MEAN_ANOMALY         12796 non-null  float64
 9   EPHEMERIS_TYPE       12796 non-null  int64  
 10  CLASSIFICATION_TYPE  12796 non-null  object 
 11  NORAD_CAT_ID         12796 non-null  int64  
 12  ELEMENT_SET_NO       12796 non-null  int64  
 13  REV_AT_EPOCH         12796 non-null  int64  
 14  BSTAR                12796 non-null  float64
 15  MEAN_MOTION_DOT      12796 non-null 

----
### Check for duplicate satellite records

In [7]:
df[df.duplicated()]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT


- This dataset is very effective, it does not have any duplicate records.
- But, when building the final pipeline, it is still necessary to handle possible duplicates
- Here is a simple method to reliablely remove duplicates

In [8]:
df = df.drop_duplicates()

-------
### Handling missing values

In [9]:
df[df.isnull()]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12791,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12792,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12793,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12794,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- By looking at the result of df.info(), there are no missing values acorss the entire dataset.
- But, df[df.isnull()] returning wierd result

  
- original dataset was "full" (had no null values),  df.isnull() mask was all False. 
- When you use this all-False mask for indexing, pandas replaces every single value with NaN, giving you a DataFrame full of NaNs.

In [10]:
# A better reliable way to check if atleast one row has misisng values or not

rows_with_null = df.isnull().any(axis=1)

In [11]:
rows_with_null

0        False
1        False
2        False
3        False
4        False
         ...  
12791    False
12792    False
12793    False
12794    False
12795    False
Length: 12796, dtype: bool

In [12]:
df[rows_with_null]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT


#### As the dataset contains no missing values, still there is a need for handling them

In [13]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-10-05T10:37:28.191360,13.762289,0.002461,90.2163,65.8752,347.2003,74.9038,0,U,900,999,3638,0.001091,1.074000e-05,0.0
1,CALSPHERE 2,1964-063E,2025-10-05T11:04:00.444864,13.528741,0.001678,90.2292,69.7816,244.2673,178.7129,0,U,902,999,82182,0.000110,8.200000e-07,0.0
2,LCS 1,1965-034C,2025-10-05T12:08:01.406976,9.893092,0.001358,32.1425,61.7207,300.3713,59.5442,0,U,1361,999,18466,-0.001468,-3.000000e-08,0.0
3,TEMPSAT 1,1965-065E,2025-10-05T06:44:31.462944,13.335738,0.006776,89.9871,212.7305,243.9611,271.6250,0,U,1512,999,92607,0.000109,6.300000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-10-05T13:13:59.657952,13.362208,0.007127,89.9109,124.8412,101.8063,291.1203,0,U,1520,999,92870,0.000297,1.650000e-06,0.0


#### We will select those features having 'logical' importance to be completlely NaNs free
Here are the following features and their methods to handle them

1. Object Name -> Fill missing names with 'Unknown'
2. Object ID -> Fill missing IDs with 'Unknown'
3. EPOCH ->  If Feature Deriving a new Feature like Days since launch -> Fill accorindgly, else drop the row
4. MEAN_MOTION,	ECCENTRICITY,	INCLINATION, RA_OF_ASC_NODE,	ARG_OF_PERICENTER,	MEAN_ANOMALY
-> mostly dropping the rows but can be derived from other features
5. NORAD_CAT_ID -> Fill with 'Unknown'
6. ELEMENT_SET_NO -> Fill with Mode
7. REV_AT_EPOCH,	BSTAR
-> Mostly imputation (mean / median)
8. MEAN_MOTION_DOT	-> Fill with (mean / median) or any other method (check carefully as it is imp feature)
9. MEAN_MOTION_DDOT -> Fill with mode (as most of them are 0)
10. BSTAR -> Handle carefully or just drop the rows

- Some other features that may not be used at all
1. EPHEMERIS_TYPE,	CLASSIFICATION_TYPE -> We will still use them by filling by 'mode' for missing values

-----

In [14]:
# Fill Missing object name
df['OBJECT_NAME'] = df['OBJECT_NAME'].fillna('Unknown')

In [15]:
# Fill Missing object ID 
df['OBJECT_ID']= df['OBJECT_ID'].fillna('Unknown')

In [16]:
# handle missing epoch -> For now, we will drop the missing rows, later perform feature engineering to fill it.
df = df.dropna(subset = ['EPOCH'])

In [17]:
# Fill Missing MEAN_MOTION, ECCENTRICITY, INCLINATION, RA_OF_ASC_NODE, ARG_OF_PERICENTER, MEAN_ANOMALY -> Drop the Nulls for now
# These are very important TLE paramters, filling them with an easay method is not recommended, as wrong value can lead to anomolous results
df = df.dropna(subset = ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY'])

In [18]:
# Fill missing NORAD_CAT_ID -> Fill with 'Unknown'
df['NORAD_CAT_ID']= df['NORAD_CAT_ID'].fillna('Unknown')

In [19]:
# Fill missing ELEMENT_SET_NO -> Fill with imputation : Mode
df['ELEMENT_SET_NO']= df['ELEMENT_SET_NO'].fillna(df['ELEMENT_SET_NO'].mode())

In [20]:
# Fill missing REV_AT_EPOCH	 -> This is also a very important feature, missing with mean or median probably is not recommended
# So, we will drop the rows right now, later we will see more secured method

df = df.dropna(subset = ['REV_AT_EPOCH'])

In [21]:
# Fill missing BSTAR -> Very important feature, so we will drop the rows to avoid anomolous filling

df = df.dropna(subset = ['BSTAR'])

In [22]:
# Fill missing MEAN_MOTION_DOT, MEAN_MOTION_DDOT -> Same like Rev at epoch and Bstar, drop the rows
df = df.dropna(subset = ['MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT'])

In [23]:
# Fill missing EPHEMERIS_TYPE, CLASSIFICATION_TYPE -> Fill each one by imputation  : Mode
df['EPHEMERIS_TYPE'] = df['EPHEMERIS_TYPE'].fillna(df['EPHEMERIS_TYPE'].mode())

df['CLASSIFICATION_TYPE'] = df['CLASSIFICATION_TYPE'].fillna(df['CLASSIFICATION_TYPE'].mode())

---
#### Save the cleaned dataset

In [24]:
df.to_csv('../data/02_cleaned/satellites_cleaned.csv', index=False)

In [25]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-10-05T10:37:28.191360,13.762289,0.002461,90.2163,65.8752,347.2003,74.9038,0,U,900,999,3638,0.001091,1.074000e-05,0.0
1,CALSPHERE 2,1964-063E,2025-10-05T11:04:00.444864,13.528741,0.001678,90.2292,69.7816,244.2673,178.7129,0,U,902,999,82182,0.000110,8.200000e-07,0.0
2,LCS 1,1965-034C,2025-10-05T12:08:01.406976,9.893092,0.001358,32.1425,61.7207,300.3713,59.5442,0,U,1361,999,18466,-0.001468,-3.000000e-08,0.0
3,TEMPSAT 1,1965-065E,2025-10-05T06:44:31.462944,13.335738,0.006776,89.9871,212.7305,243.9611,271.6250,0,U,1512,999,92607,0.000109,6.300000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-10-05T13:13:59.657952,13.362208,0.007127,89.9109,124.8412,101.8063,291.1203,0,U,1520,999,92870,0.000297,1.650000e-06,0.0
